In [1]:
import networkx as nx
from mupt.mupr.topology import TopologicalStructure

from mupt.builders.dpd import DPDRandomWalk
from mupt.geometry.coordinates.directions import random_unit_vector

ModuleNotFoundError: No module named 'anytree'

In [ ]:
#system parameters
R_excl : float = 10.0
bond_length : float = 1.5 # 5.5
angle_max_rad : float = np.pi/4

n_chains : int = 4
chain_len_min : int = 10
chain_len_max : int = 15

In [ ]:
univprim = Primitive(label='universe')
with build_progress() as progress:
    task = progress.add_task('Building chains', total=n_chains, chain_len=0)
    for chain_len in np.random.randint(chain_len_min, chain_len_max + 1, size=n_chains):
        progress.update(task, chain_len=chain_len)
        
        # build chain hierarchy
        molprim = Primitive(label=f'{chain_len}-mer_chain')
        for i, unit_name in enumerate(
            sequence_repeat_units(
                chain_len,
                head_name='head',
                tail_name='tail',
                mid_distrib=rep_unit_distrib_mid,
            )
        ):
            rep_unit_prim = lexicon[unit_name].copy()
            molprim.attach_child(rep_unit_prim)
            
        # assign path graph as topology, connecting adjacent Primitives
        molprim.set_topology(
            nx.path_graph(
                molprim.children_by_handle.keys(),
                create_using=TopologicalStructure,
            ),
            max_registration_iter=100,
        )

        # place beads by random walk (other build implementations would go here)
        direction = random_unit_vector()
        builder = DPDRandomWalk()
        for handle, placement in builder.generate_placements(molprim):
            molprim.children_by_handle[handle].rigidly_transform(placement)
            
        # attach chain to universe, expand to show topology of beads
        mol_handle = univprim.attach_child(molprim)
        # univprim.expand(mol_handle)
        
        progress.advance(task)
        progress.refresh()

univprim.visualize_topology()
print(univprim.hierarchy_summary(to_depth=2))